<a href="https://www.kaggle.com/code/rohinipandit/notebookb990caa941?scriptVersionId=306080948" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import pandas as pd
import os

# Find the file
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/kandeelai22/messy-e-commerce-sales-dataset/messy_ecommerce_sales_data.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Check data types and missing values
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

In [ ]:
# 1. Fix column name spaces
df.columns = df.columns.str.strip()

# 2. Remove duplicate rows
df = df.drop_duplicates()

# 3. Fix data types
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# 4. Fill missing Category with 'Unknown'
df['Category'] = df['Category'].fillna('Unknown')

# 5. Fill missing Quantity and Price with median
df['Quantity'] = df['Quantity'].fillna(df['Quantity'].median())
df['Price'] = df['Price'].fillna(df['Price'].median())

# 6. Recalculate Total where missing
df['Total'] = df['Total'].fillna(df['Quantity'] * df['Price'])

# Verify
print("Missing values after cleaning:")
print(df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("\nShape:", df.shape)

In [ ]:
# Drop rows where Order_Date couldn't be parsed
df = df.dropna(subset=['Order_Date'])

# Extract useful date features
df['Month'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.strftime('%b')
df['Year'] = df['Order_Date'].dt.year

print("Final Shape:", df.shape)
print("\nCleaning Complete! ✅")
df.head()

In [ ]:
# Sales by Category
plt.figure(figsize=(8,4))
category_sales = df.groupby('Category')['Total'].sum().sort_values(ascending=False)
sns.barplot(x=category_sales.values, y=category_sales.index, palette='viridis')
plt.title('Total Revenue by Category')
plt.xlabel('Total Revenue')
plt.tight_layout()
plt.show()

print(category_sales)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sales by Category
plt.figure(figsize=(8,4))
category_sales = df.groupby('Category')['Total'].sum().sort_values(ascending=False)
sns.barplot(x=category_sales.values, y=category_sales.index, palette='viridis')
plt.title('Total Revenue by Category')
plt.xlabel('Total Revenue')
plt.tight_layout()
plt.show()

print(category_sales)

In [ ]:
# Fix inconsistent category names
df['Category'] = df['Category'].str.strip().str.title()

# Remove rows with negative Total (corrupted data)
df = df[df['Total'] >= 0]

print("Categories after fixing:")
print(df['Category'].value_counts())
print("\nShape:", df.shape)

In [ ]:
# Merge 'Electronic' into 'Electronics'
df['Category'] = df['Category'].replace('Electronic', 'Electronics')

print("Final categories:")
print(df['Category'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('E-Commerce Sales Analysis', fontsize=14, fontweight='bold')

# Chart 1: Revenue by Category
category_sales = df.groupby('Category')['Total'].sum().sort_values(ascending=False)
axes[0,0].barh(category_sales.index, category_sales.values, color='steelblue')
axes[0,0].set_title('Revenue by Category')
axes[0,0].set_xlabel('Total Revenue')

# Chart 2: Orders by Payment Method
payment_counts = df['Payment_Method'].value_counts()
axes[0,1].pie(payment_counts.values, labels=payment_counts.index, autopct='%1.1f%%')
axes[0,1].set_title('Payment Methods')

# Chart 3: Monthly Revenue Trend
monthly = df.groupby('Month')['Total'].sum()
axes[1,0].plot(monthly.index, monthly.values, marker='o', color='green')
axes[1,0].set_title('Monthly Revenue Trend')
axes[1,0].set_xlabel('Month')
axes[1,0].set_ylabel('Revenue')

# Chart 4: Order Status Distribution
status_counts = df['Status'].value_counts()
axes[1,1].bar(status_counts.index, status_counts.values, color='coral')
axes[1,1].set_title('Order Status')
axes[1,1].set_xlabel('Status')

plt.tight_layout()
plt.show()

In [ ]:
print("=== KEY INSIGHTS ===\n")

# Insight 1
top_cat = df.groupby('Category')['Total'].sum().idxmax()
top_rev = df.groupby('Category')['Total'].sum().max()
print(f"1. Top Revenue Category: {top_cat} (₹{top_rev:,.2f})")

# Insight 2
top_payment = df['Payment_Method'].value_counts().idxmax()
top_pct = df['Payment_Method'].value_counts(normalize=True).max() * 100
print(f"2. Most Used Payment Method: {top_payment} ({top_pct:.1f}% of orders)")

# Insight 3
best_month = df.groupby('Month')['Total'].sum().idxmax()
print(f"3. Best Sales Month: Month {best_month}")

# Insight 4
delivered = df[df['Status']=='Delivered'].shape[0]
total_orders = df.shape[0]
print(f"4. Delivery Rate: {delivered}/{total_orders} orders delivered ({delivered/total_orders*100:.1f}%)")

# Insight 5
avg_order = df['Total'].mean()
print(f"5. Average Order Value: ₹{avg_order:,.2f}")

In [ ]:
## Step 5: Summary & Conclusions

### Dataset
- Started with 103 rows, ended with 96 clean rows after removing duplicates, bad dates, and corrupted values

### Cleaning Steps Performed
- Stripped leading spaces from column names
- Removed 1 duplicate row
- Fixed data types: Order_Date (datetime), Quantity & Price (numeric)
- Filled missing Category with 'Unknown', Quantity & Price with median
- Recalculated missing Total values
- Standardized inconsistent category names (Electronics, Sports)
- Removed 1 row with negative Total (-196741) — likely data entry error

### Key Business Insights
1. **Books** is the top revenue category (₹40,296)
2. **Cash on Delivery** is the most preferred payment method (31.2%)
3. **February** had the highest sales — possible seasonal spike
4. **Delivery rate is only 10.4%** — majority of orders are in Processing/Shipped/Returned/Cancelled status. This is a serious operational concern.
5. **Average order value is ₹1,612** — mid-range basket size

### Tools Used
- Python, Pandas, Matplotlib, Kaggle Notebooks

## Step 5: Summary & Conclusions

### Dataset
- Started with 103 rows, ended with 96 clean rows after removing duplicates, bad dates, and corrupted values

### Cleaning Steps Performed
- Stripped leading spaces from column names
- Removed 1 duplicate row
- Fixed data types: Order_Date (datetime), Quantity & Price (numeric)
- Filled missing Category with 'Unknown', Quantity & Price with median
- Recalculated missing Total values
- Standardized inconsistent category names (Electronics, Sports)
- Removed 1 row with negative Total (-196741) — likely data entry error

### Key Business Insights
1. **Books** is the top revenue category (₹40,296)
2. **Cash on Delivery** is the most preferred payment method (31.2%)
3. **February** had the highest sales — possible seasonal spike
4. **Delivery rate is only 10.4%** — majority of orders are in Processing/Shipped/Returned/Cancelled status. This is a serious operational concern.
5. **Average order value is ₹1,612** — mid-range basket size

### Tools Used
- Python, Pandas, Matplotlib, Kaggle Notebooks